# 🔧 Notebook 2 — Feature Engineering
**Auto Insurance Churn Project**

### What is Feature Engineering?
Raw data is rarely in the right shape for a machine learning model.
Feature engineering is the process of transforming raw columns into
meaningful signals the model can learn from.

Examples of what we do here:
- `days_tenure` (a raw number) → `tenure_bucket` (a lifecycle stage)
- `home_market_value` (a string range) → a numeric midpoint the model can use
- `curr_ann_amt / income` → a premium-to-income ratio (price pressure proxy)

Good features = better models, even with simpler algorithms.


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from data_loader import load_main, save_processed
from features import build_features, get_model_features

df_raw = load_main(nrows=200_000)
print(f"Raw dataset: {df_raw.shape}")


## 1. Apply Feature Engineering Pipeline

In [ ]:
# build_features() applies all transforms in sequence
# Open src/features.py to see each function explained
df = build_features(df_raw)

print("New columns added:")
new_cols = [c for c in df.columns if c not in df_raw.columns]
print(new_cols)
df[new_cols + ["churn"]].head(5)


## 2. Tenure Bucket Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume by bucket
tenure_counts = df["tenure_bucket"].value_counts()
axes[0].bar(tenure_counts.index, tenure_counts.values, color="steelblue")
axes[0].set_title("Customer Count by Tenure Bucket")
axes[0].set_ylabel("Count")

# Churn rate by bucket
tenure_churn = df.groupby("tenure_bucket", observed=True)["churn"].mean() * 100
axes[1].bar(tenure_churn.index, tenure_churn.values, color="tomato")
axes[1].set_title("Churn Rate by Tenure Bucket")
axes[1].set_ylabel("Churn Rate (%)")
for i, v in enumerate(tenure_churn.values):
    axes[1].text(i, v + 0.1, f"{v:.1f}%", ha="center")

plt.suptitle("Tenure Bucket Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/02_tenure_buckets.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Premium-to-Income Ratio

Customers spending a larger share of income on insurance are under more financial pressure and may be more likely to cancel.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df[df["churn"]==0]["premium_income_ratio"].clip(0, 0.05).plot(
    kind="kde", label="Retained", color="steelblue", ax=ax)
df[df["churn"]==1]["premium_income_ratio"].clip(0, 0.05).plot(
    kind="kde", label="Churned", color="tomato", ax=ax)
ax.set_title("Premium-to-Income Ratio by Churn Status", fontsize=13, fontweight="bold")
ax.set_xlabel("Annual Premium / Annual Income")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/02_premium_income_ratio.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Handle Missing Values

In [ ]:
# home_market_value_num will be NaN where original string didn't map
# Fill with median — a safe imputation for tree-based models
df["home_market_value_num"] = df["home_market_value_num"].fillna(df["home_market_value_num"].median())
df["premium_income_ratio"] = df["premium_income_ratio"].fillna(df["premium_income_ratio"].median())

print("Missing values after imputation:")
print(df[get_model_features()].isnull().sum())


## 5. Prepare Final Model Dataset

In [ ]:
feature_cols = get_model_features()
target_col = "churn"

model_df = df[feature_cols + [target_col]].dropna()
print(f"Model-ready dataset: {model_df.shape}")
print(f"Churn rate: {model_df[target_col].mean()*100:.2f}%")

# Save to data/processed/ for the modeling notebook
save_processed(model_df, "churn_model_ready.csv")
model_df.describe()


**Next:** `03_modeling.ipynb` — train Logistic Regression and XGBoost, evaluate with ROC-AUC, and explain predictions with SHAP.